In [ ]:
import json
import torch
import librosa
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

SR = 16000

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
w2v_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
w2v_model.eval()

with open("end-data/manifest.json") as f:
    samples = [json.loads(line) for line in f if line.strip()]

for s in samples:
    s.setdefault("text", "")
    s.setdefault("end_type", "unknown")

    pred_path = s["predicted_filepath"]
    s["pred_audio"], _ = librosa.load(pred_path, sr=SR)
    # Durations are always computed from loaded audio — never trust manifest "duration" field.
    s["pred_duration"] = len(s["pred_audio"]) / SR

    if s.get("audio_filepath", "").strip():
        try:
            s["gt_audio"], _ = librosa.load(s["audio_filepath"].strip(), sr=SR)
        except Exception as e:
            print(f"Could not load GT for {pred_path}: {e}")
            s["gt_audio"] = None
    else:
        s["gt_audio"] = None

print(f"Loaded {len(samples)} samples:")
for i, s in enumerate(samples):
    gt_dur = f"{len(s['gt_audio'])/SR:.2f}s" if s['gt_audio'] is not None else "N/A"
    print(f"  [{i}] end_type={s['end_type']:8s} | pred={s['pred_duration']:.2f}s | gt={gt_dur} | text: {s['text']}")

## 1. Listen to all samples (predicted vs ground truth)

In [ ]:
for i, s in enumerate(samples):
    print(f"=== Sample {i}: end_type='{s['end_type']}' ===")
    print(f"Text: {s['text']}")
    print(f"Predicted ({len(s['pred_audio'])/SR:.2f}s):")
    ipd.display(ipd.Audio(s['pred_audio'], rate=SR, normalize=False))
    if s['gt_audio'] is not None:
        print(f"Ground truth ({len(s['gt_audio'])/SR:.2f}s):")
        ipd.display(ipd.Audio(s['gt_audio'], rate=SR, normalize=False))
    else:
        print("Ground truth: not available")
    print()

## 2. Waveform and spectrogram comparison (predicted vs ground truth)

In [ ]:
def plot_waveform_and_spec(audio, sr, title, ax_wave, ax_spec):
    t = np.arange(len(audio)) / sr
    ax_wave.plot(t, audio, linewidth=0.3)
    ax_wave.set_title(f"{title} — Waveform")
    ax_wave.set_xlabel("Time (s)")
    ax_wave.set_ylabel("Amplitude")
    ax_wave.set_xlim(0, t[-1])

    S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80, fmax=8000)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel', ax=ax_spec, fmax=8000)
    ax_spec.set_title(f"{title} — Mel Spectrogram")

for i, s in enumerate(samples):
    has_gt = s['gt_audio'] is not None
    nrows = 4 if has_gt else 2
    fig, axes = plt.subplots(nrows, 1, figsize=(14, 3 * nrows))

    plot_waveform_and_spec(s['pred_audio'], SR, f"[{i}] Predicted (end_type='{s['end_type']}')", axes[0], axes[1])
    if has_gt:
        plot_waveform_and_spec(s['gt_audio'], SR, f"[{i}] Ground Truth (clean)", axes[2], axes[3])

    fig.suptitle(f"Sample {i}: \"{s['text']}\"", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 3. Tail analysis — zoom into the last 1 second

We look at energy envelope, spectral centroid, and zero-crossing rate in the tail region to see if these simple features separate the end types.

In [ ]:
TAIL_SEC = 1.0
HOP_LENGTH = 256
FRAME_LENGTH = 1024

def compute_tail_features(audio, sr, tail_sec=TAIL_SEC):
    tail_samples = int(tail_sec * sr)
    tail = audio[-tail_samples:]
    full = audio

    rms_full = librosa.feature.rms(y=full, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    rms_tail = rms_full[-len(rms_full) * tail_samples // len(full):]

    zcr_full = librosa.feature.zero_crossing_rate(y=full, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    zcr_tail = zcr_full[-len(zcr_full) * tail_samples // len(full):]

    centroid_full = librosa.feature.spectral_centroid(y=full, sr=sr, hop_length=HOP_LENGTH)[0]
    centroid_tail = centroid_full[-len(centroid_full) * tail_samples // len(full):]

    return {
        'tail': tail, 'rms_full': rms_full, 'rms_tail': rms_tail,
        'zcr_full': zcr_full, 'zcr_tail': zcr_tail,
        'centroid_full': centroid_full, 'centroid_tail': centroid_tail,
    }

for i, s in enumerate(samples):
    feats = compute_tail_features(s['pred_audio'], SR)

    fig, axes = plt.subplots(3, 2, figsize=(16, 9))
    fig.suptitle(f"Sample {i}: end_type='{s['end_type']}' — \"{s['text']}\"", fontsize=13)

    t_full = librosa.frames_to_time(np.arange(len(feats['rms_full'])), sr=SR, hop_length=HOP_LENGTH)
    t_tail = np.linspace(len(s['pred_audio'])/SR - TAIL_SEC, len(s['pred_audio'])/SR, len(feats['rms_tail']))

    # Full signal RMS
    axes[0, 0].plot(t_full, feats['rms_full'], color='steelblue')
    axes[0, 0].set_title("RMS Energy (full)")
    axes[0, 0].axvline(len(s['pred_audio'])/SR - TAIL_SEC, color='red', ls='--', label='tail start')
    axes[0, 0].legend()

    # Tail RMS
    axes[0, 1].plot(t_tail, feats['rms_tail'], color='steelblue')
    axes[0, 1].set_title("RMS Energy (last 1s)")
    axes[0, 1].fill_between(t_tail, feats['rms_tail'], alpha=0.3)

    # Full ZCR
    axes[1, 0].plot(t_full, feats['zcr_full'], color='darkorange')
    axes[1, 0].set_title("Zero-Crossing Rate (full)")
    axes[1, 0].axvline(len(s['pred_audio'])/SR - TAIL_SEC, color='red', ls='--')

    # Tail ZCR
    axes[1, 1].plot(t_tail, feats['zcr_tail'], color='darkorange')
    axes[1, 1].set_title("Zero-Crossing Rate (last 1s)")

    # Full spectral centroid
    axes[2, 0].plot(t_full, feats['centroid_full'], color='seagreen')
    axes[2, 0].set_title("Spectral Centroid (full)")
    axes[2, 0].axvline(len(s['pred_audio'])/SR - TAIL_SEC, color='red', ls='--')

    # Tail spectral centroid
    axes[2, 1].plot(t_tail, feats['centroid_tail'], color='seagreen')
    axes[2, 1].set_title("Spectral Centroid (last 1s)")

    for ax in axes.flat:
        ax.set_xlabel("Time (s)")

    plt.tight_layout()
    plt.show()

    # Also listen to just the tail
    tail_audio = s['pred_audio'][-int(TAIL_SEC * SR):]
    print(f"  Tail audio (last {TAIL_SEC}s):")
    ipd.display(ipd.Audio(tail_audio, rate=SR, normalize=False))
    print()

## 4. Wav2Vec2 forced alignment — frame-level speech boundary detection

We use `torchaudio.functional.forced_align` to align the known target text against the Wav2Vec2 CTC emission. Unlike greedy CTC decoding (which tends to "give up" a few frames early at speech boundaries), forced alignment assigns every target token to specific frames, giving tighter boundary estimates. If text is unavailable or alignment fails (e.g., cutoff audio too short for the full text), we fall back to greedy decoding.

In [ ]:
import torchaudio.functional as taf

BLANK_TOKEN_ID = processor.tokenizer.pad_token_id  # CTC blank
W2V_VOCAB = processor.tokenizer.get_vocab()

def text_to_tokens(text):
    """Convert text to wav2vec2 character token IDs for forced alignment."""
    text = text.upper().strip()
    tokens = []
    for i, word in enumerate(text.split()):
        if i > 0:
            tokens.append(W2V_VOCAB['|'])
        for char in word:
            if char in W2V_VOCAB:
                tokens.append(W2V_VOCAB[char])
    return tokens

def _build_segments(aligned_ids, scores, frame_duration):
    """Build token segments from frame-level alignment IDs and scores."""
    segments = []
    current_token = int(aligned_ids[0])
    start_frame = 0
    for idx in range(1, len(aligned_ids)):
        tid = int(aligned_ids[idx])
        if tid != current_token:
            seg_scores = scores[start_frame:idx]
            segments.append({
                'token': processor.decode([current_token]),
                'token_id': current_token,
                'is_blank': (current_token == BLANK_TOKEN_ID),
                'start_time': start_frame * frame_duration,
                'end_time': idx * frame_duration,
                'duration': (idx - start_frame) * frame_duration,
                'mean_confidence': float(seg_scores.mean()),
                'min_confidence': float(seg_scores.min()),
            })
            start_frame = idx
            current_token = tid
    seg_scores = scores[start_frame:]
    segments.append({
        'token': processor.decode([current_token]),
        'token_id': current_token,
        'is_blank': (current_token == BLANK_TOKEN_ID),
        'start_time': start_frame * frame_duration,
        'end_time': len(aligned_ids) * frame_duration,
        'duration': (len(aligned_ids) - start_frame) * frame_duration,
        'mean_confidence': float(seg_scores.mean()),
        'min_confidence': float(seg_scores.min()),
    })
    return segments

def speech_analysis(audio, sr, text=""):
    """Analyze speech boundaries using forced alignment (preferred) or greedy CTC fallback."""
    input_values = processor(audio, return_tensors="pt", sampling_rate=sr).input_values
    with torch.no_grad():
        logits = w2v_model(input_values).logits[0]

    log_probs = torch.log_softmax(logits, dim=-1)
    probs = torch.exp(log_probs)
    n = len(logits)
    frame_duration = len(audio) / n / sr

    max_probs = probs.max(dim=-1).values.numpy()
    blank_probs = probs[:, BLANK_TOKEN_ID].numpy()
    greedy_ids = torch.argmax(logits, dim=-1).numpy()
    greedy_text = processor.decode(greedy_ids, skip_special_tokens=True)

    method = "greedy"
    aligned_ids = greedy_ids
    alignment_scores = max_probs

    if text.strip():
        target_tokens = text_to_tokens(text)
        if target_tokens:
            try:
                fa_ids, fa_scores = taf.forced_align(
                    log_probs.unsqueeze(0),
                    torch.tensor([target_tokens]),
                    blank=BLANK_TOKEN_ID,
                )
                aligned_ids = fa_ids[0].numpy()
                alignment_scores = torch.exp(fa_scores[0]).numpy()
                method = "forced"
            except Exception as e:
                print(f"  [!] Forced alignment failed ({e}), using greedy CTC")

    segments = _build_segments(aligned_ids, alignment_scores, frame_duration)
    speech_segments = [s for s in segments if not s['is_blank']]
    speech_end = speech_segments[-1]['end_time'] if speech_segments else 0.0

    # Per-frame RMS energy (used by feature extraction, not for boundary correction)
    frame_rms = np.zeros(n)
    for fi in range(n):
        s0 = int(fi * frame_duration * sr)
        s1 = min(int((fi + 1) * frame_duration * sr), len(audio))
        if s1 > s0:
            frame_rms[fi] = np.sqrt(np.mean(audio[s0:s1] ** 2))

    speech_mask = aligned_ids != BLANK_TOKEN_ID
    speech_rms_mean = frame_rms[speech_mask].mean() if speech_mask.any() else 1e-10

    return {
        'method': method,
        'segments': segments,
        'speech_segments': speech_segments,
        'greedy_text': greedy_text,
        'frame_duration': frame_duration,
        'max_probs': max_probs,
        'blank_probs': blank_probs,
        'aligned_ids': aligned_ids,
        'n_frames': n,
        'frame_rms': frame_rms,
        'speech_rms_mean': speech_rms_mean,
        'speech_end': speech_end,
    }

for i, s in enumerate(samples):
    result = speech_analysis(s['pred_audio'], SR, text=s.get('text', ''))
    s['ctc'] = result

    print(f"\n{'='*80}")
    print(f"Sample {i}: end_type='{s['end_type']}'  [method={result['method']}]")
    print(f"Target text:  {s['text']}")
    print(f"Greedy CTC:   {result['greedy_text']}")
    print(f"Frames: {result['n_frames']}, frame_dur: {result['frame_duration']*1000:.1f}ms")
    print(f"\nLast 8 segments:")
    for seg in result['segments'][-8:]:
        marker = "     " if seg['is_blank'] else " >>> "
        print(f"  {marker} '{seg['token']:3s}' | {seg['start_time']:.3f}-{seg['end_time']:.3f}s "
              f"(dur={seg['duration']:.3f}s) | conf={seg['mean_confidence']:.3f}")

    audio_end = len(s['pred_audio']) / SR
    speech_end = result['speech_end']
    print(f"\nSpeech end:  {speech_end:.3f}s  (trailing: {audio_end - speech_end:.3f}s)")
    print(f"Audio end:   {audio_end:.3f}s")

### Visualize CTC frame probabilities and blank/speech regions

In [ ]:
for i, s in enumerate(samples):
    ctc = s['ctc']
    fd = ctc['frame_duration']
    t_frames = np.arange(ctc['n_frames']) * fd
    speech_end = ctc['speech_end']
    method_label = f"[{ctc['method']}]"

    fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f"Sample {i}: end_type='{s['end_type']}' — \"{s['text']}\" — {method_label}", fontsize=13)

    t_audio = np.arange(len(s['pred_audio'])) / SR
    axes[0].plot(t_audio, s['pred_audio'], linewidth=0.3, color='gray')
    for seg in ctc['speech_segments']:
        axes[0].axvspan(seg['start_time'], seg['end_time'], alpha=0.3, color='steelblue')
    axes[0].axvline(speech_end, color='red', ls='--', lw=2, label=f"Speech end @ {speech_end:.3f}s")
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title("Waveform with aligned speech regions (blue)")

    axes[1].bar(t_frames, ctc['frame_rms'], width=fd*0.9, color='steelblue', alpha=0.7, label='Frame RMS')
    axes[1].axvline(speech_end, color='red', ls='--', lw=1.5)
    axes[1].set_ylabel("RMS Energy")
    axes[1].set_title("Per-frame audio energy")
    axes[1].legend(fontsize=9)

    axes[2].plot(t_frames, ctc['max_probs'], linewidth=0.5, color='steelblue', label='Max token prob')
    axes[2].plot(t_frames, ctc['blank_probs'], linewidth=0.5, color='tomato', alpha=0.7, label='Blank prob')
    axes[2].axvline(speech_end, color='red', ls='--', lw=1.5)
    axes[2].set_ylabel("Probability")
    axes[2].set_title("Per-frame CTC probabilities")
    axes[2].legend(fontsize=9)
    axes[2].set_ylim(0, 1.05)

    is_speech_frame = ctc['aligned_ids'] != BLANK_TOKEN_ID
    axes[3].fill_between(t_frames, 0, is_speech_frame.astype(float), alpha=0.5, color='steelblue', label='Speech frame')
    axes[3].fill_between(t_frames, 0, (~is_speech_frame).astype(float) * -1, alpha=0.3, color='tomato', label='Blank frame')
    axes[3].axvline(speech_end, color='red', ls='--', lw=1.5, label='Speech end')
    axes[3].set_ylabel("Speech / Blank")
    axes[3].set_xlabel("Time (s)")
    axes[3].set_title("Aligned speech vs blank frames")
    axes[3].legend(fontsize=9)
    axes[3].set_ylim(-1.1, 1.1)

    plt.tight_layout()
    plt.show()

## 5. Feature extraction for end-type classification

We extract a battery of features from each sample that might help distinguish end types:
- **Trailing duration**: time from last CTC speech frame to audio end
- **Tail RMS energy**: mean energy in the trailing region
- **Tail RMS ratio**: ratio of trailing energy to overall energy
- **Last speech segment confidence**: how confident the model is about the last token
- **Last speech segment duration**: unnaturally short = cutoff, long = noise/hallucination
- **Tail spectral centroid**: noise tends to have different spectral content
- **Tail ZCR**: high ZCR after speech can indicate noise
- **Energy drop rate**: how quickly energy falls at the end

In [ ]:
def extract_end_features(audio, ctc, sr=SR, end_type=None):
    """Extract end-detection features from audio + its alignment analysis."""
    audio_dur = len(audio) / sr

    features = {'audio_duration': audio_dur}
    if end_type is not None:
        features['end_type'] = end_type

    if ctc['speech_segments']:
        last_speech = ctc['speech_segments'][-1]
        features['speech_end'] = ctc['speech_end']
        features['last_token'] = last_speech['token']
        features['last_token_duration'] = last_speech['duration']
        features['last_token_confidence'] = last_speech['mean_confidence']
        features['last_token_min_confidence'] = last_speech['min_confidence']
    else:
        features['speech_end'] = 0
        features['last_token'] = ''
        features['last_token_duration'] = 0
        features['last_token_confidence'] = 0
        features['last_token_min_confidence'] = 0

    speech_end = features['speech_end']
    features['trailing_duration'] = audio_dur - speech_end
    features['trailing_fraction'] = features['trailing_duration'] / audio_dur

    trail_start = int(speech_end * sr)
    trailing_audio = audio[trail_start:]

    if len(trailing_audio) > 0:
        rms_trail = np.sqrt(np.mean(trailing_audio ** 2))
        rms_full = np.sqrt(np.mean(audio ** 2))
        features['trail_rms'] = rms_trail
        features['trail_rms_ratio'] = rms_trail / (rms_full + 1e-10)
    else:
        features['trail_rms'] = 0
        features['trail_rms_ratio'] = 0

    window_samples = int(0.2 * sr)
    if len(audio) > 2 * window_samples:
        rms_last = np.sqrt(np.mean(audio[-window_samples:] ** 2))
        rms_prev = np.sqrt(np.mean(audio[-2*window_samples:-window_samples] ** 2))
        features['energy_drop_ratio'] = rms_last / (rms_prev + 1e-10)
    else:
        features['energy_drop_ratio'] = 1.0

    if len(trailing_audio) > FRAME_LENGTH:
        centroid = librosa.feature.spectral_centroid(y=trailing_audio, sr=sr, hop_length=HOP_LENGTH)
        zcr = librosa.feature.zero_crossing_rate(y=trailing_audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)
        features['trail_spectral_centroid'] = float(centroid.mean())
        features['trail_zcr'] = float(zcr.mean())
    else:
        features['trail_spectral_centroid'] = 0
        features['trail_zcr'] = 0

    if ctc['speech_segments']:
        last_frame = int(speech_end / ctc['frame_duration'])
        trail_blank_probs = ctc['blank_probs'][last_frame:]
        features['trail_mean_blank_prob'] = float(trail_blank_probs.mean()) if len(trail_blank_probs) > 0 else 1.0
        aligned_ids = ctc['aligned_ids']
        trail_speech_count = (aligned_ids[last_frame:] != BLANK_TOKEN_ID).sum()
        features['trail_speech_frame_fraction'] = float(trail_speech_count) / max(len(trail_blank_probs), 1)
    else:
        features['trail_mean_blank_prob'] = 1.0
        features['trail_speech_frame_fraction'] = 0

    return features

all_features = []
for i, s in enumerate(samples):
    feats = extract_end_features(s['pred_audio'], s['ctc'], end_type=s['end_type'])
    all_features.append(feats)
    print(f"\n--- Sample {i}: end_type='{feats['end_type']}' ---")
    for k, v in feats.items():
        if k == 'end_type':
            continue
        if isinstance(v, float):
            print(f"  {k:35s} = {v:.4f}")
        else:
            print(f"  {k:35s} = {v}")

## 6. Compare features across end types

Bar charts and scatter plots to see which features separate the classes. With only 3 samples this is purely exploratory, but it helps identify which signals look promising.

In [ ]:
import pandas as pd

df = pd.DataFrame(all_features)
display(df)

color_map = {'clean': 'seagreen', 'cutoff': 'tomato', 'noise': 'darkorange', 'silence': 'slategray'}
colors = [color_map.get(t, 'gray') for t in df['end_type']]

numeric_cols = [
    'trailing_duration', 'trailing_fraction', 'last_token_duration',
    'last_token_confidence', 'trail_rms', 'trail_rms_ratio',
    'energy_drop_ratio', 'trail_spectral_centroid', 'trail_zcr',
    'trail_mean_blank_prob', 'trail_speech_frame_fraction',
]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flat

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    bars = ax.bar(range(len(df)), df[col], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels([f"{r['end_type']}" for _, r in df.iterrows()], rotation=30, ha='right')
    ax.set_title(col, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

# Hide unused axes
for idx in range(len(numeric_cols), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Feature comparison across end types", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Scatter plots: pairwise feature relationships

Look at 2D scatter plots of the most promising feature pairs to see if end types cluster.

In [ ]:
scatter_pairs = [
    ('trailing_duration', 'trail_rms_ratio'),
    ('trailing_duration', 'trail_speech_frame_fraction'),
    ('last_token_confidence', 'last_token_duration'),
    ('energy_drop_ratio', 'trail_zcr'),
    ('trail_spectral_centroid', 'trail_rms'),
    ('trailing_fraction', 'trail_mean_blank_prob'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flat

for idx, (xf, yf) in enumerate(scatter_pairs):
    ax = axes[idx]
    for _, row in df.iterrows():
        c = color_map.get(row['end_type'], 'gray')
        ax.scatter(row[xf], row[yf], c=c, s=150, edgecolors='black', linewidth=1, zorder=3)
        ax.annotate(row['end_type'], (row[xf], row[yf]), fontsize=9,
                    xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel(xf)
    ax.set_ylabel(yf)
    ax.grid(alpha=0.3)

fig.suptitle("Pairwise feature scatter plots (colored by end_type)", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Proposed heuristic classifier and evaluation

Based on the features above, we sketch a rule-based classifier and show its predictions alongside the ground truth labels. This can be refined once more data is available.

In [ ]:
def classify_end_type(feats):
    """
    Heuristic classifier based on CTC + energy features.
    Thresholds are rough and need tuning with more data.
    
    Decision tree:
    1. Cutoff: last CTC token very short + small trailing region
    2. Noise: significant trailing region with high energy relative to speech
    3. Silence: significant trailing region with very low energy
    4. Clean: everything else (moderate trailing, natural energy decay)
    """
    trailing = feats['trailing_duration']
    trail_rms = feats['trail_rms_ratio']
    trail_speech_frac = feats['trail_speech_frame_fraction']
    last_dur = feats['last_token_duration']
    last_conf = feats['last_token_confidence']
    energy_drop = feats['energy_drop_ratio']

    # Cutoff: speech is forced all the way to the audio end, AND the last
    # aligned token looks suspicious (short duration and/or low confidence).
    # Using AND vs OR here is TBD — see diagnostic table below.
    if trailing < 0.06 and last_dur < 0.025 and last_conf < 0.1:
        return 'cutoff'

    # Noise: significant trailing region with high energy (relative to overall)
    # trail_rms_ratio > 0.5 means trailing region is at least half as loud as the speech
    if trailing > 0.3 and trail_rms > 0.5:
        return 'noise'

    # Silence: very long trailing region with low energy relative to speech.
    # trail_rms_ratio up to ~0.10 still reads as silence — quantization noise
    # and room tone in a long trailing region can push the ratio above 0.05
    # even when the signal is perceptually silent.
    if trailing > 1.0 and trail_rms < 0.10:
        return 'silence'

    return 'clean'

# --- Summary table with key signals ---
print(f"{'#':>3s}  {'True':>10s}  {'Predicted':>10s}  {'OK?':>4s}  {'trail_s':>7s}  {'rms_ratio':>9s}  {'last_dur':>8s}  {'last_conf':>9s}")
print("-" * 75)
correct = 0
for i, feats in enumerate(all_features):
    pred = classify_end_type(feats)
    ok = pred == feats['end_type']
    correct += ok
    print(f"{i:>3d}  {feats['end_type']:>10s}  {pred:>10s}  {'OK' if ok else 'MISS':>4s}  "
          f"{feats['trailing_duration']:7.3f}  {feats['trail_rms_ratio']:9.4f}  "
          f"{feats['last_token_duration']:8.3f}  {feats['last_token_confidence']:9.3f}")
total = len(all_features)
print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.0f}%)")

# --- Cutoff rule variant diagnostic ---
TRAIL_THRESH = 0.06
DUR_THRESH = 0.025
CONF_THRESH = 0.1

print(f"\n{'='*80}")
print("Cutoff rule diagnostic — AND vs OR")
print(f"  Thresholds: trailing<{TRAIL_THRESH}, last_dur<{DUR_THRESH}, last_conf<{CONF_THRESH}")
print(f"{'#':>3s}  {'True':>10s}  {'trailing':>9s}  {'last_dur':>9s}  {'last_conf':>10s}  "
      f"{'trail?':>6s}  {'dur?':>5s}  {'conf?':>5s}  {'AND':>5s}  {'OR':>5s}")
print("-" * 80)
for i, feats in enumerate(all_features):
    tr = feats['trailing_duration']
    ld = feats['last_token_duration']
    lc = feats['last_token_confidence']
    t_fire = tr < TRAIL_THRESH
    d_fire = ld < DUR_THRESH
    c_fire = lc < CONF_THRESH
    and_rule = t_fire and d_fire and c_fire
    or_rule  = t_fire and (d_fire or c_fire)
    print(f"{i:>3d}  {feats['end_type']:>10s}  {tr:9.3f}  {ld:9.3f}  {lc:10.3f}  "
          f"{'Y' if t_fire else '.':>6s}  {'Y' if d_fire else '.':>5s}  {'Y' if c_fire else '.':>5s}  "
          f"{'CUT' if and_rule else '.':>5s}  {'CUT' if or_rule else '.':>5s}")
print("\nAND = requires all three; OR = requires trailing + either duration or confidence.")

## 8b. Sanity check: run the same pipeline on ground truth audio

Ground truth audio should always classify as "clean". If the classifier disagrees, that reveals threshold issues. We also compare GT feature values side-by-side with predicted features to understand the baseline.

In [ ]:
gt_features = []
for i, s in enumerate(samples):
    if s['gt_audio'] is None:
        print(f"Sample {i}: GT audio not available, skipping.")
        gt_features.append(None)
        continue

    gt_ctc = speech_analysis(s['gt_audio'], SR, text=s.get('text', ''))
    s['gt_ctc'] = gt_ctc
    gt_feats = extract_end_features(s['gt_audio'], gt_ctc, end_type='clean')
    gt_features.append(gt_feats)

    print(f"\n{'='*80}")
    print(f"Sample {i} — Ground Truth (expected: clean)  [method={gt_ctc['method']}]")
    print(f"Target text:  {s['text']}")
    print(f"Greedy CTC:   {gt_ctc['greedy_text']}")
    if gt_ctc['speech_segments']:
        last = gt_ctc['speech_segments'][-1]
        print(f"Last speech token: '{last['token']}' ends at {last['end_time']:.3f}s")
        print(f"Audio ends at {len(s['gt_audio'])/SR:.3f}s -> trailing: {gt_feats['trailing_duration']:.3f}s")

    pred_label = classify_end_type(gt_feats)
    status = "OK" if pred_label == 'clean' else f"WRONG (got '{pred_label}')"
    print(f"Classifier prediction: {pred_label} — {status}")

print(f"\n\n{'='*80}")
print("SUMMARY: Ground truth classification (all should be 'clean')")
print(f"{'#':>3s}  {'Expected':>10s}  {'Predicted':>10s}  {'OK?':>4s}  {'trail_s':>7s}  {'rms_ratio':>9s}  {'last_dur':>8s}  {'last_conf':>9s}")
print("-" * 75)
gt_correct = 0
gt_total = 0
for i, gf in enumerate(gt_features):
    if gf is None:
        print(f"{i:>3d}  {'clean':>10s}  {'N/A':>10s}  skip")
        continue
    gt_total += 1
    pred = classify_end_type(gf)
    ok = pred == 'clean'
    gt_correct += ok
    print(f"{i:>3d}  {'clean':>10s}  {pred:>10s}  {'OK' if ok else 'MISS':>4s}  "
          f"{gf['trailing_duration']:7.3f}  {gf['trail_rms_ratio']:9.4f}  "
          f"{gf['last_token_duration']:8.3f}  {gf['last_token_confidence']:9.3f}")
if gt_total > 0:
    print(f"\nAccuracy: {gt_correct}/{gt_total} ({100*gt_correct/gt_total:.0f}%)")
else:
    print("\nNo GT samples available.")

### Side-by-side feature comparison: predicted vs ground truth

In [ ]:
compare_keys = [
    'trailing_duration', 'trailing_fraction',
    'last_token_duration', 'last_token_confidence', 'trail_rms', 'trail_rms_ratio',
    'energy_drop_ratio', 'trail_spectral_centroid', 'trail_zcr',
    'trail_mean_blank_prob', 'trail_speech_frame_fraction',
]

for i, (pf, gf) in enumerate(zip(all_features, gt_features)):
    if gf is None:
        print(f"\nSample {i}: GT not available")
        continue
    print(f"\n--- Sample {i}: pred end_type='{pf['end_type']}' ---")
    print(f"  {'Feature':35s}  {'Predicted':>12s}  {'GT (clean)':>12s}  {'Ratio P/GT':>12s}")
    print(f"  {'-'*75}")
    for k in compare_keys:
        pv = pf.get(k, 0)
        gv = gf.get(k, 0)
        ratio = pv / gv if gv != 0 else float('inf') if pv != 0 else 1.0
        print(f"  {k:35s}  {pv:12.4f}  {gv:12.4f}  {ratio:12.2f}")

In [ ]:
plot_keys = [
    'trailing_duration', 'trail_rms_ratio', 'last_token_duration',
    'last_token_confidence', 'energy_drop_ratio', 'trail_zcr',
]

valid_pairs = [(i, pf, gf) for i, (pf, gf) in enumerate(zip(all_features, gt_features)) if gf is not None]

fig, axes = plt.subplots(len(valid_pairs), len(plot_keys), figsize=(18, 4 * len(valid_pairs)))
if len(valid_pairs) == 1:
    axes = axes[np.newaxis, :]

for row, (i, pf, gf) in enumerate(valid_pairs):
    for col, k in enumerate(plot_keys):
        ax = axes[row, col]
        pv, gv = pf.get(k, 0), gf.get(k, 0)
        bars = ax.bar(['Predicted', 'GT (clean)'], [pv, gv],
                       color=[color_map.get(pf['end_type'], 'gray'), 'seagreen'],
                       edgecolor='black', linewidth=0.5)
        ax.set_title(f"{k}" if row == 0 else k, fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        if col == 0:
            ax.set_ylabel(f"Sample {i}\n({pf['end_type']})", fontsize=10)

fig.suptitle("Feature comparison: Predicted vs Ground Truth (clean)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Compare predicted audio tail with ground truth tail

Since ground truth is always "clean", comparing the energy/spectral profile of the predicted tail against the GT tail can help calibrate what "normal" looks like.

In [ ]:
for i, s in enumerate(samples):
    if s['gt_audio'] is None:
        print(f"Sample {i}: GT not available, skipping.")
        continue

    gt_ctc = speech_analysis(s['gt_audio'], SR, text=s.get('text', ''))

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    fig.suptitle(f"Sample {i}: \"{s['text']}\" — Predicted (end_type='{s['end_type']}') vs Ground Truth (clean)", fontsize=13)

    # RMS comparison
    pred_rms = librosa.feature.rms(y=s['pred_audio'], frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    gt_rms = librosa.feature.rms(y=s['gt_audio'], frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]

    t_pred_rms = librosa.frames_to_time(np.arange(len(pred_rms)), sr=SR, hop_length=HOP_LENGTH)
    t_gt_rms = librosa.frames_to_time(np.arange(len(gt_rms)), sr=SR, hop_length=HOP_LENGTH)

    # Align from end: show last 2 seconds
    axes[0, 0].plot(t_pred_rms - t_pred_rms[-1], pred_rms, label='Predicted', color='tomato')
    axes[0, 0].plot(t_gt_rms - t_gt_rms[-1], gt_rms, label='Ground Truth', color='seagreen', alpha=0.7)
    axes[0, 0].set_title("RMS Energy (aligned from end)")
    axes[0, 0].set_xlabel("Time relative to end (s)")
    axes[0, 0].set_xlim(-2, 0)
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    # CTC blank probability comparison (aligned from end)
    pred_fd = s['ctc']['frame_duration']
    gt_fd = gt_ctc['frame_duration']
    t_pred_ctc = np.arange(s['ctc']['n_frames']) * pred_fd
    t_gt_ctc = np.arange(gt_ctc['n_frames']) * gt_fd

    axes[0, 1].plot(t_pred_ctc - t_pred_ctc[-1] * pred_fd / pred_fd, s['ctc']['blank_probs'],
                    label='Predicted', color='tomato', linewidth=0.5)
    axes[0, 1].plot(t_gt_ctc - t_gt_ctc[-1] * gt_fd / gt_fd, gt_ctc['blank_probs'],
                    label='Ground Truth', color='seagreen', alpha=0.7, linewidth=0.5)
    axes[0, 1].set_title("CTC Blank Probability (aligned from end)")
    axes[0, 1].set_xlim(-2, 0)
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    # Spectrograms of last 1s, side by side
    tail_pred = s['pred_audio'][-int(1.0 * SR):]
    tail_gt = s['gt_audio'][-int(1.0 * SR):]

    S_pred = librosa.feature.melspectrogram(y=tail_pred, sr=SR, n_mels=80, fmax=8000)
    S_gt = librosa.feature.melspectrogram(y=tail_gt, sr=SR, n_mels=80, fmax=8000)

    librosa.display.specshow(librosa.power_to_db(S_pred, ref=np.max), sr=SR, x_axis='time',
                             y_axis='mel', ax=axes[1, 0], fmax=8000)
    axes[1, 0].set_title("Predicted — Mel spectrogram (last 1s)")

    librosa.display.specshow(librosa.power_to_db(S_gt, ref=np.max), sr=SR, x_axis='time',
                             y_axis='mel', ax=axes[1, 1], fmax=8000)
    axes[1, 1].set_title("Ground Truth — Mel spectrogram (last 1s)")

    plt.tight_layout()
    plt.show()

    # Listen to tails side by side
    print(f"  Predicted tail (last 1s):")
    ipd.display(ipd.Audio(tail_pred, rate=SR, normalize=False))
    print(f"  Ground Truth tail (last 1s):")
    ipd.display(ipd.Audio(tail_gt, rate=SR, normalize=False))
    print()

## 10. Summary and next steps

**What the forced alignment approach gives us:**
- Frame-level speech boundary detection using `torchaudio.functional.forced_align` against the known target text
- More precise speech-end timestamps than greedy CTC (which tends to "give up" a few frames early)
- No energy-correction hacks needed — the alignment is forced to cover all target tokens
- Falls back to greedy CTC if text is unavailable or alignment fails (e.g., cutoff audio too short)

**Promising features so far:**
- `trailing_duration` and `trail_rms_ratio` together separate silence (long trailing, low energy) from noise (long trailing, high energy)
- `last_token_duration` and `last_token_confidence` help detect cutoff vs clean endings
- `energy_drop_ratio` captures how abruptly the signal ends

**Limitations:**
- Thresholds in the heuristic classifier need tuning with more labeled data
- Forced alignment assumes the audio contains the target text — for badly hallucinated outputs, greedy fallback is used

**Suggested next steps:**
1. Collect more labeled examples (especially silence and noise)
2. Train a small classifier (e.g., logistic regression or decision tree) on the extracted features
3. Try stronger forced aligners (e.g., Montreal Forced Aligner) for even tighter boundaries